**Тема «POS-tagger и NER»**

Задание 1.

Написать теггер на данных с русским языком:
1. проверить UnigramTagger, BigramTagger, TrigramTagger и их комбинации;
2. написать свой теггер как на занятии, попробовать разные векторайзеры, добавить знание не только букв, но и слов;
3. сравнить все реализованные методы, сделать выводы.


Задание 2.

Проверить, насколько хорошо работает NER
(Данные брать из Index of /pub/named_entities):
1. проверить NER из nltk/spacy/deeppavlov;
2. написать свой NER, попробовать разные подходы.

1) передаём в сетку токен и его соседей.

2) передаём в сетку только токен.

3) свой вариант.
3. сравнить свои реализованные подходы на качество — вывести precision/recall/f1_score.


In [1]:
from google.colab import drive
drive.mount('./drive')

Drive already mounted at ./drive; to attempt to forcibly remount, call drive.mount("./drive", force_remount=True).


In [2]:
!pip -q install pyconll
!pip -q install corus
!pip -q install -U spacy
!pip3 -q install deeppavlov

In [3]:
!wget -O ru_syntagrus-ud-train.conllu https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-a.conllu
!wget -O ru_syntagrus-ud-dev.conllu https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-dev.conllu

--2023-12-04 19:15:18--  https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-a.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 40736599 (39M) [text/plain]
Saving to: ‘ru_syntagrus-ud-train.conllu’

ru_syntagrus-ud-tra 100%[===================>]  38.85M   191MB/s    in 0.2s    

2023-12-04 19:15:19 (191 MB/s) - ‘ru_syntagrus-ud-train.conllu’ saved [40736599/40736599]

--2023-12-04 19:15:19--  https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-dev.conllu
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111

In [56]:
import pandas as pd
import numpy as np
import re

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn import model_selection, preprocessing, linear_model

import pyconll
import nltk
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger, TrigramTagger, RegexpTagger
from nltk.tokenize import word_tokenize

import corus
from corus import load_ne5

import spacy
from spacy import displacy
import ru_core_news_sm
from spacy.lang.ru.examples import sentences
from spacy.lang.ru import Russian

import tensorflow as tf
from keras import Sequential
from keras.layers import Dense, Embedding, GlobalAveragePooling1D
from keras.layers import GlobalMaxPooling1D, Conv1D, GRU, LSTM, Dropout, Input
from keras.layers.experimental.preprocessing import TextVectorization

from razdel import tokenize
from corus import load_ne5

from razdel import tokenize
from corus import load_ne5

import matplotlib
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [6]:
data_train = pyconll.load_from_file('/content/drive/MyDrive/NLP/ru_syntagrus-ud-train.conllu')
data_test = pyconll.load_from_file('/content/drive/MyDrive/NLP/ru_syntagrus-ud-dev.conllu')

In [7]:
fdata_train = []
for sent in data_train[:]:
    fdata_train.append([(token.form, token.upos) for token in sent])

fdata_test = []
for sent in data_test[:]:
    fdata_test.append([(token.form, token.upos) for token in sent])

fdata_sent_test = []
for sent in data_test[:]:
    fdata_sent_test.append([token.form for token in sent])

In [8]:
len(fdata_train), len(fdata_test), len(fdata_sent_test)

(24516, 8906, 8906)

In [9]:
fdata_train[:2]

[[('Анкета', 'NOUN'), ('.', 'PUNCT')],
 [('Начальник', 'NOUN'),
  ('областного', 'ADJ'),
  ('управления', 'NOUN'),
  ('связи', 'NOUN'),
  ('Семен', 'PROPN'),
  ('Еремеевич', 'PROPN'),
  ('был', 'AUX'),
  ('человек', 'NOUN'),
  ('простой', 'ADJ'),
  (',', 'PUNCT'),
  ('приходил', 'VERB'),
  ('на', 'ADP'),
  ('работу', 'NOUN'),
  ('всегда', 'ADV'),
  ('вовремя', 'ADV'),
  (',', 'PUNCT'),
  ('здоровался', 'VERB'),
  ('с', 'ADP'),
  ('секретаршей', 'NOUN'),
  ('за', 'ADP'),
  ('руку', 'NOUN'),
  ('и', 'CCONJ'),
  ('иногда', 'ADV'),
  ('даже', 'PART'),
  ('писал', 'VERB'),
  ('в', 'ADP'),
  ('стенгазету', 'NOUN'),
  ('заметки', 'NOUN'),
  ('под', 'ADP'),
  ('псевдонимом', 'NOUN'),
  ('"', 'PUNCT'),
  ('Муха', 'NOUN'),
  ('"', 'PUNCT'),
  ('.', 'PUNCT')]]

In [11]:
default_tagger = nltk.DefaultTagger('NN')
default_acc = default_tagger.evaluate(fdata_test)

unigram_tagger = UnigramTagger(fdata_train)
unigram_acc = unigram_tagger.evaluate(fdata_test)

bigram_tagger = BigramTagger(fdata_train)
bigram_acc = bigram_tagger.evaluate(fdata_test)

trigram_tagger = TrigramTagger(fdata_train)
trigram_acc = trigram_tagger.evaluate(fdata_test)

bigram_tagger = BigramTagger(fdata_train, backoff=unigram_tagger)
bigram_unigram_acc = bigram_tagger.evaluate(fdata_test)

trigram_tagger = TrigramTagger(fdata_train, backoff=bigram_tagger)
trigram_bigram_unigram_acc = trigram_tagger.evaluate(fdata_test)

print(f'Accuracy:\nDefault Tagger: {round(default_acc, 3)},\nUnigram Tagger: {round(unigram_acc, 3)},\nBigram Tagger: {round(bigram_acc, 5)},\n'
      f'Trigram Tagger: {round(trigram_acc, 3)},\nBigram and Unigram Tagger: {round(bigram_unigram_acc, 5)},\n'
      f'Trigram, Bigram and Unigram Tagger: {round(trigram_bigram_unigram_acc, 5)},\n')

Accuracy:
Default Tagger: 0.0,
Unigram Tagger: 0.824,
Bigram Tagger: 0.60939,
Trigram Tagger: 0.178,
Bigram and Unigram Tagger: 0.82928,
Trigram, Bigram and Unigram Tagger: 0.82914,



Различные комбинации теггеров могут давать прирост качества.

Попробуем объединить работу всех теггеров с помощью функции.

In [12]:
def union_taggers(train_sents, tagger_classes, backoff=None):
    for cls in tagger_classes:
        backoff = cls(train_sents, backoff=backoff)
    return backoff


backoff = DefaultTagger('NN')
tag = union_taggers(fdata_train,
                     [UnigramTagger, BigramTagger, TrigramTagger],
                     backoff = backoff)

tag.evaluate(fdata_test)

0.827905462595221

Получили усредненный результат. Не самый высокий.

Преобразуем тренировочный датасет в списки слов и списки POS-разметки.

In [14]:
train_tok = []
train_label = []
for sent in fdata_train[:]:
    for tok in sent:
        train_tok.append(tok[0])
        train_label.append('NO_TAG' if tok[1] is None else tok[1])

test_tok = []
test_label = []
for sent in fdata_test[:]:
    for tok in sent:
        test_tok.append(' ' if tok[0] is None else tok[0])
        test_label.append('NO_TAG' if tok[1] is None else tok[1])

In [15]:
train_tok[:7], train_label[:7]

(['Анкета', '.', 'Начальник', 'областного', 'управления', 'связи', 'Семен'],
 ['NOUN', 'PUNCT', 'NOUN', 'ADJ', 'NOUN', 'NOUN', 'PROPN'])

In [16]:
le = LabelEncoder()
train_enc_labels = le.fit_transform(train_label)
train_enc_labels

array([ 7, 13,  7, ...,  1, 11, 13])

In [17]:
test_enc_labels = le.transform(test_label)
test_enc_labels

array([ 7, 13,  1, ...,  0,  7, 13])

In [18]:
le.classes_

array(['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN',
       'NO_TAG', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM',
       'VERB', 'X'], dtype='<U6')

In [19]:
%time

vectorizers = [CountVectorizer(ngram_range=(1, 5), analyzer='char'),
               TfidfVectorizer(ngram_range=(1, 5), analyzer='char'),
               HashingVectorizer(ngram_range=(1, 5), analyzer='char', n_features=1000)]
vectorizers_word = [CountVectorizer(ngram_range=(1, 5), analyzer='word'),
               TfidfVectorizer(ngram_range=(1, 5), analyzer='word'),
               HashingVectorizer(ngram_range=(1, 5), analyzer='word', n_features=1000)]
n_features = [2000, 3000, 5000]
vectorizers_hash = [HashingVectorizer(ngram_range=(1, 5), analyzer='char', n_features=feat) for feat in n_features]
vectorizers_hash_word = [HashingVectorizer(ngram_range=(1, 5), analyzer='word', n_features=feat) for feat in n_features]
f1_scores = []
accuracy_scores = []

for vectorizer in vectorizers + vectorizers_word + vectorizers_hash + vectorizers_hash_word:
    X_train = vectorizer.fit_transform(train_tok)
    X_test = vectorizer.transform(test_tok)

    lr = LogisticRegression(random_state=0, max_iter=100)
    lr.fit(X_train, train_enc_labels)
    pred = lr.predict(X_test)
    f1 = f1_score(test_enc_labels, pred, average='weighted')
    f1_scores.append(f1)
    acc = accuracy_score(test_enc_labels, pred)
    accuracy_scores.append(acc)

    print(vectorizer)
    print(classification_report(test_enc_labels, pred, target_names=le.classes_))

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 8.11 µs
CountVectorizer(analyzer='char', ngram_range=(1, 5))
              precision    recall  f1-score   support

         ADJ       0.93      0.93      0.93     15103
         ADP       0.98      1.00      0.99     13717
         ADV       0.92      0.92      0.92      7783
         AUX       0.82      0.96      0.88      1390
       CCONJ       0.89      0.97      0.93      5672
         DET       0.89      0.70      0.79      4265
        INTJ       0.39      0.29      0.33        24
        NOUN       0.94      0.97      0.96     36238
      NO_TAG       1.00      0.77      0.87       265
         NUM       0.86      0.91      0.88      1734
        PART       0.95      0.77      0.85      5125
        PRON       0.86      0.89      0.87      7444
       PROPN       0.83      0.66      0.74      5473
       PUNCT       1.00      1.00      1.00     29186
       SCONJ       0.75      0.97      0.85      2865
         SYM      

In [21]:
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('max_colwidth', None)

In [22]:
result_model = pd.DataFrame({'Vectorizer': vectorizers + vectorizers_word + vectorizers_hash + vectorizers_hash_word,
                            'f1_score': f1_scores})
result_model.sort_values('f1_score', ascending=False)

,Vectorizer,f1_score
0,"CountVectorizer(analyzer='char', ngram_range=(1, 5))",0.937951
1,"TfidfVectorizer(analyzer='char', ngram_range=(1, 5))",0.923645
8,"HashingVectorizer(analyzer='char', n_features=5000, ngram_range=(1, 5))",0.909227
7,"HashingVectorizer(analyzer='char', n_features=3000, ngram_range=(1, 5))",0.899331
6,"HashingVectorizer(analyzer='char', n_features=2000, ngram_range=(1, 5))",0.891962
2,"HashingVectorizer(analyzer='char', n_features=1000, ngram_range=(1, 5))",0.864832
4,"TfidfVectorizer(ngram_range=(1, 5))",0.649657
3,"CountVectorizer(ngram_range=(1, 5))",0.638731
11,"HashingVectorizer(n_features=5000, ngram_range=(1, 5))",0.585121
10,"HashingVectorizer(n_features=3000, ngram_range=(1, 5))",0.565790


In [23]:
result_model_acc = pd.DataFrame({'Vectorizer': vectorizers + vectorizers_word + vectorizers_hash + vectorizers_hash_word,
                            'Accuracy': accuracy_scores})
result_model_acc.sort_values('Accuracy', ascending=False)

,Vectorizer,Accuracy
0,"CountVectorizer(analyzer='char', ngram_range=(1, 5))",0.939456
1,"TfidfVectorizer(analyzer='char', ngram_range=(1, 5))",0.926818
8,"HashingVectorizer(analyzer='char', n_features=5000, ngram_range=(1, 5))",0.912403
7,"HashingVectorizer(analyzer='char', n_features=3000, ngram_range=(1, 5))",0.903067
6,"HashingVectorizer(analyzer='char', n_features=2000, ngram_range=(1, 5))",0.895540
2,"HashingVectorizer(analyzer='char', n_features=1000, ngram_range=(1, 5))",0.868650
4,"TfidfVectorizer(ngram_range=(1, 5))",0.639306
3,"CountVectorizer(ngram_range=(1, 5))",0.627821
11,"HashingVectorizer(n_features=5000, ngram_range=(1, 5))",0.603848
10,"HashingVectorizer(n_features=3000, ngram_range=(1, 5))",0.589661


В результате эксперимента проверили различные векторайзеры для разного количества символов. Наилучший результат показали символьные N-граммы. Результат N-грамм для слов оказался значительно хуже.

Задание 2.

NLTK

In [24]:
nltk.download('tagsets')
nltk.download('averaged_perceptron_tagger')
nltk.download('maxent_ne_chunker')
nltk.download('words')

[nltk_data] Downloading package tagsets to /root/nltk_data...
[nltk_data]   Unzipping help/tagsets.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [25]:
nltk.help.upenn_tagset('RB')
nltk.help.upenn_tagset('NN')
nltk.help.upenn_tagset('VB')

RB: adverb
    occasionally unabatingly maddeningly adventurously professedly
    stirringly prominently technologically magisterially predominately
    swiftly fiscally pitilessly ...
NN: noun, common, singular or mass
    common-carrier cabbage knuckle-duster Casino afghan shed thermostat
    investment slide humour falloff slick wind hyena override subhumanity
    machinist ...
VB: verb, base form
    ask assemble assess assign assume atone attention avoid bake balkanize
    bank begin behold believe bend benefit bevel beware bless boil bomb
    boost brace break bring broil brush build ...


In [26]:
!wget http://www.labinform.ru/pub/named_entities/collection5.zip

--2023-12-04 19:56:48--  http://www.labinform.ru/pub/named_entities/collection5.zip
Resolving www.labinform.ru (www.labinform.ru)... 95.181.230.181
Connecting to www.labinform.ru (www.labinform.ru)|95.181.230.181|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1899530 (1.8M) [application/zip]
Saving to: ‘collection5.zip’

collection5.zip     100%[===================>]   1.81M  2.00MB/s    in 0.9s    

2023-12-04 19:56:49 (2.00 MB/s) - ‘collection5.zip’ saved [1899530/1899530]



In [27]:
!unzip collection5.zip

Archive:  collection5.zip
   creating: Collection5/
  inflating: Collection5/001.ann     
  inflating: Collection5/001.txt     
  inflating: Collection5/002.ann     
  inflating: Collection5/002.txt     
  inflating: Collection5/003.ann     
  inflating: Collection5/003.txt     
  inflating: Collection5/004.ann     
  inflating: Collection5/004.txt     
  inflating: Collection5/005.ann     
  inflating: Collection5/005.txt     
  inflating: Collection5/006.ann     
  inflating: Collection5/006.txt     
  inflating: Collection5/007.ann     
  inflating: Collection5/007.txt     
  inflating: Collection5/008.ann     
  inflating: Collection5/008.txt     
  inflating: Collection5/009.ann     
  inflating: Collection5/009.txt     
  inflating: Collection5/010.ann     
  inflating: Collection5/010.txt     
  inflating: Collection5/011.ann     
  inflating: Collection5/011.txt     
  inflating: Collection5/012.ann     
  inflating: Collection5/012.txt     
  inflating: Collection5/013.ann    

In [28]:
records = load_ne5('Collection5/')

In [29]:
document = next(records)
text = document.text
# text
text = re.sub('\r\n\r\n',' ',text)
text = re.sub('\r\n',' ',text)
text

'Губернатор Югры назначила нового заместителя по строительству и ЖКХ  Губернатор Ханты-Мансийского автономного округа - Югры Наталья Комарова назначила на должность своего заместителя Дмитрия Шаповала; в его ведении будут находиться окружные департаменты дорожного хозяйства и транспорта, строительства, ЖКХ и энергетики. Губернатор Ханты-Мансийского автономного округа – Югры Наталья Комарова. Архив  Губернатор Ханты-Мансийского автономного округа — Югры Наталья Комарова назначила на должность своего заместителя Дмитрия Шаповала; в его ведении будут находиться окружные департаменты дорожного хозяйства и транспорта, строительства, ЖКХ и энергетики, сообщает пресс-служба губернатора. "Новым заместителем главы региона с 21 июня 2013 года назначен Дмитрий Викторович Шаповал, который будет отвечать за окружные департаменты дорожного хозяйства и транспорта, строительства, ЖКХ и энергетики, гражданской защиты населения, а также служба государственного надзора за техническим состоянием самоходны

In [30]:
document.text = text

In [32]:
nltk.download('punkt')
nltk.pos_tag(nltk.word_tokenize(text)) # Как предварительно очистить все статьи в словаре.

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


[('Губернатор', 'JJ'),
 ('Югры', 'NNP'),
 ('назначила', 'NNP'),
 ('нового', 'NNP'),
 ('заместителя', 'NNP'),
 ('по', 'NNP'),
 ('строительству', 'NNP'),
 ('и', 'NNP'),
 ('ЖКХ', 'NNP'),
 ('Губернатор', 'NNP'),
 ('Ханты-Мансийского', 'JJ'),
 ('автономного', 'NNP'),
 ('округа', 'NNP'),
 ('-', ':'),
 ('Югры', 'NN'),
 ('Наталья', 'JJ'),
 ('Комарова', 'NNP'),
 ('назначила', 'NNP'),
 ('на', 'NNP'),
 ('должность', 'NNP'),
 ('своего', 'NNP'),
 ('заместителя', 'NNP'),
 ('Дмитрия', 'NNP'),
 ('Шаповала', 'NNP'),
 (';', ':'),
 ('в', 'NNP'),
 ('его', 'NNP'),
 ('ведении', 'NNP'),
 ('будут', 'NNP'),
 ('находиться', 'NNP'),
 ('окружные', 'NNP'),
 ('департаменты', 'NNP'),
 ('дорожного', 'NNP'),
 ('хозяйства', 'NNP'),
 ('и', 'NNP'),
 ('транспорта', 'NNP'),
 (',', ','),
 ('строительства', 'NNP'),
 (',', ','),
 ('ЖКХ', 'NNP'),
 ('и', 'NNP'),
 ('энергетики', 'NNP'),
 ('.', '.'),
 ('Губернатор', 'VB'),
 ('Ханты-Мансийского', 'JJ'),
 ('автономного', 'NNP'),
 ('округа', 'NNP'),
 ('–', 'NNP'),
 ('Югры', 'NNP'),


In [33]:
{(' '.join(c[0] for c in chunk), chunk.label() ) for chunk in nltk.ne_chunk(nltk.pos_tag(nltk.word_tokenize(text))) if hasattr(chunk, 'label') }

{('Губернатор', 'ORGANIZATION'),
 ('Губернатор', 'PERSON'),
 ('Дмитрий Викторович Шаповал', 'PERSON'),
 ('Дмитрия Шаповала', 'PERSON'),
 ('ЖКХ', 'ORGANIZATION'),
 ('Магнитогорский', 'PERSON'),
 ('Наталья Комарова', 'PERSON'),
 ('Носова', 'PERSON'),
 ('Шаповал', 'PERSON'),
 ('Югры', 'ORGANIZATION'),
 ('Югры Наталья Комарова', 'PERSON')}

В целом - работает. Но ошибки встречаются.

In [34]:
document.spans

[Ne5Span(
     index='T1',
     type='LOC',
     start=11,
     stop=15,
     text='Югры'
 ),
 Ne5Span(
     index='T2',
     type='LOC',
     start=84,
     stop=127,
     text='Ханты-Мансийского автономного округа - Югры'
 ),
 Ne5Span(
     index='T3',
     type='PER',
     start=128,
     stop=144,
     text='Наталья Комарова'
 ),
 Ne5Span(
     index='T4',
     type='PER',
     start=187,
     stop=203,
     text='Дмитрия Шаповала'
 ),
 Ne5Span(
     index='T5',
     type='LOC',
     start=337,
     stop=380,
     text='Ханты-Мансийского автономного округа – Югры'
 ),
 Ne5Span(
     index='T6',
     type='PER',
     start=381,
     stop=397,
     text='Наталья Комарова'
 ),
 Ne5Span(
     index='T7',
     type='LOC',
     start=420,
     stop=463,
     text='Ханты-Мансийского автономного округа — Югры'
 ),
 Ne5Span(
     index='T8',
     type='PER',
     start=464,
     stop=480,
     text='Наталья Комарова'
 ),
 Ne5Span(
     index='T9',
     type='PER',
     start=523,
     stop=

SpaCy

In [35]:
!python -m spacy download ru_core_news_sm

2023-12-04 19:58:37.822573: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-12-04 19:58:37.822674: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-12-04 19:58:37.822732: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-12-04 19:58:39.588665: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 47.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')


In [36]:
nlp = spacy.load('ru_core_news_sm')

In [37]:
ny_bb = text
article = nlp(ny_bb)

In [38]:
displacy.render(article, jupyter=True, style='ent')

На этом тексте библиотека SpaCy без ошибок.

Посмотрим на список токенов, частей речи и сущностей.

In [39]:
for token in article:
    print(token.text, token.pos_, token.dep_)

Губернатор NOUN nsubj
Югры PROPN nmod
назначила VERB ROOT
нового ADJ amod
заместителя NOUN obj
по ADP case
строительству NOUN nmod
и CCONJ cc
ЖКХ PROPN conj
  SPACE dep
Губернатор NOUN appos
Ханты ADJ amod
- ADJ amod
Мансийского ADJ amod
автономного ADJ amod
округа NOUN nmod
- NOUN nmod
Югры PROPN nmod
Наталья PROPN appos
Комарова PROPN flat:name
назначила VERB xcomp
на ADP case
должность NOUN obl
своего DET det
заместителя NOUN nmod
Дмитрия PROPN appos
Шаповала PROPN flat:name
; PUNCT punct
в ADP case
его DET det
ведении NOUN obl
будут AUX aux
находиться VERB conj
окружные ADJ amod
департаменты NOUN nsubj
дорожного ADJ amod
хозяйства NOUN nmod
и CCONJ cc
транспорта NOUN conj
, PUNCT punct
строительства NOUN conj
, PUNCT punct
ЖКХ PROPN conj
и CCONJ cc
энергетики NOUN conj
. PUNCT punct
Губернатор NOUN ROOT
Ханты ADJ amod
- ADJ amod
Мансийского ADJ amod
автономного ADJ amod
округа NOUN nmod
– PUNCT punct
Югры PROPN parataxis
Наталья PROPN appos
Комарова PROPN flat:name
. PUNCT punct
Ар

DeepPavlov

In [47]:
!python -m venv env
!.\env\Scripts\activate.bat
!pip install deeppavlov
!python -m deeppavlov install squad_bert
!python -m deeppavlov install ner_ontonotes

import deeppavlov
from deeppavlov import configs, build_model
deeppavlov_ner = build_model(configs.ner.ner_rus, download=True)
rus_document = "Нью-Йорк, США, 30 апреля 2020, 01:01 — REGNUM В администрации президента США Дональда Трампа планируют пройти все этапы создания вакцины от коронавируса в ускоренном темпе и выпустить 100 млн доз до конца 2020 года, передаёт агентство Bloomberg со ссылкой на осведомлённые источники"
deeppavlov_ner([rus_document])

The virtual environment was not created successfully because ensurepip is not
available.  On Debian/Ubuntu systems, you need to install the python3-venv
package using the following command.

    apt install python3.10-venv

You may need to use sudo with that command.  After installing the python3-venv
package, recreate your virtual environment.

Failing command: /content/env/bin/python3

/bin/bash: line 1: .envScriptsactivate.bat: command not found
Ignoring transformers: markers 'python_version < "3.8"' don't match your environment
2023-12-04 20:07:46.601 WARNING in 'deeppavlov.core.common.file'['file'] at line 39: 

################################################################################
# The model 'ner_ontonotes' has been removed from the DeepPavlov configs.
# The model 'ner_ontonotes_bert' is used instead.
# To disable this message please switch to 'ner_ontonotes_bert'.
# Automatic name resolving will be disabled in the deeppavlov 1.2.0,
# and if you try to use 'ner_ontonot

AttributeError: ignored

Не получается установить deeppavlov.

Самописный NER.

In [50]:
def get_classification_report(y_test_true, y_test_pred):
    print(classification_report(y_test_true, y_test_pred))

    print('CONFUSION MATRIX\n')
    print(pd.crosstab(y_test_true, y_test_pred))

Воспользуемся размеченным корпусом текстов.

In [51]:
records = load_ne5('Collection5/')
next(records)

Ne5Markup(
    id='1016',
    text='Губернатор Югры назначила нового заместителя по строительству и ЖКХ\r\n\r\n\r\nГубернатор Ханты-Мансийского автономного округа - Югры Наталья Комарова назначила на должность своего заместителя Дмитрия Шаповала; в его ведении будут находиться окружные департаменты дорожного хозяйства и транспорта, строительства, ЖКХ и энергетики.\r\nГубернатор Ханты-Мансийского автономного округа – Югры Наталья Комарова. Архив\r\n\r\n Губернатор Ханты-Мансийского автономного округа — Югры Наталья Комарова назначила на должность своего заместителя Дмитрия Шаповала; в его ведении будут находиться окружные департаменты дорожного хозяйства и транспорта, строительства, ЖКХ и энергетики, сообщает пресс-служба губернатора.\r\n\r\n"Новым заместителем главы региона с 21 июня 2013 года назначен Дмитрий Викторович Шаповал, который будет отвечать за окружные департаменты дорожного хозяйства и транспорта, строительства, ЖКХ и энергетики, гражданской защиты населения, а также служб

In [52]:
words_docs = []
for ix, rec in enumerate(records):
    words = []
    for token in tokenize(rec.text):
        type_ent = 'OUT'
        for ent in rec.spans:
            if (token.start >= ent.start) and (token.stop <= ent.stop):
                type_ent = ent.type
                break
        words.append([token.text, type_ent])
    words_docs.extend(words)

In [53]:
df_words = pd.DataFrame(words_docs, columns=['word', 'tag'])

In [54]:
df_words['tag'].value_counts()

OUT         218990
PER          21183
ORG          13644
LOC           4547
GEOPOLIT      4354
MEDIA         2482
Name: tag, dtype: int64

In [55]:
df_words.head()

,word,tag
0,Директора,OUT
1,МВФ,ORG
2,назначили,OUT
3,арбитром,OUT
4,межгосударственных,OUT


In [57]:
train_x, valid_x, train_y, valid_y = model_selection.train_test_split(df_words['word'], df_words['tag'])

Закодируем целевую переменную.

In [58]:
encoder = preprocessing.LabelEncoder()
train_y = encoder.fit_transform(train_y)
valid_y = encoder.fit_transform(valid_y)

Посмотрим на классы.

In [59]:
encoder.classes_

array(['GEOPOLIT', 'LOC', 'MEDIA', 'ORG', 'OUT', 'PER'], dtype=object)

In [60]:
train_x.apply(len).max(axis=0)

55

In [61]:
valid_x

228773    кандидатуру
174558              М
108780     обеспечить
35484               "
29653       прокуроре
             ...     
29297               ,
104583    спутниковые
103396           года
188866          также
237984       пообещал
Name: word, Length: 66300, dtype: object

In [62]:
train_data = tf.data.Dataset.from_tensor_slices((train_x, train_y))
valid_data = tf.data.Dataset.from_tensor_slices((valid_x, valid_y))

train_data = train_data.batch(16)
valid_data = valid_data.batch(16)

In [63]:
AUTOTUNE = tf.data.experimental.AUTOTUNE

train_data = train_data.cache().prefetch(buffer_size=AUTOTUNE)
valid_data = valid_data.cache().prefetch(buffer_size=AUTOTUNE)

In [64]:
def custom_standardization(input_data):
    # Здесь может быть предобработка текста
    return input_data

vocab_size = 30000
seq_len = 10

vectorize_layer = TextVectorization(
                            standardize=custom_standardization,
                            max_tokens=vocab_size,
                            output_mode='int',
                            #ngrams=(1, 3),
                            output_sequence_length=seq_len)

# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
text_data = train_data.map(lambda x, y: x)
vectorize_layer.adapt(text_data)

In [65]:
len(vectorize_layer.get_vocabulary())

29872

In [66]:
embedding_dim = 64

class modelNER(tf.keras.Model):
    def __init__(self):
        super(modelNER, self).__init__()
        self.emb = Embedding(vocab_size, embedding_dim)
        self.gPool = GlobalMaxPooling1D()
        self.fc1 = Dense(300, activation='relu')
        self.fc2 = Dense(50, activation='relu')
        self.fc3 = Dense(6, activation='softmax') # [OUT, PER, ORG, LOC, GEOPOLIT, MEDIA]

    def call(self, x):
        x = vectorize_layer(x)
        x = self.emb(x)
        pool_x = self.gPool(x)

        fc_x = self.fc1(pool_x)
        fc_x = self.fc2(fc_x)

        concat_x = tf.concat([pool_x, fc_x], axis=1)
        prob = self.fc3(concat_x)
        return prob

In [67]:
mmodel = modelNER()

In [68]:
mmodel.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [69]:
mmodel.fit( train_data,
            validation_data=valid_data,
            epochs=3)

Epoch 1/3
12432/12432 [==============================] - 553s 44ms/step - loss: 0.2975 - accuracy: 0.9127 - val_loss: 0.2087 - val_accuracy: 0.9370
Epoch 2/3
12432/12432 [==============================] - 498s 40ms/step - loss: 0.1264 - accuracy: 0.9624 - val_loss: 0.2074 - val_accuracy: 0.9415
Epoch 3/3
12432/12432 [==============================] - 484s 39ms/step - loss: 0.1099 - accuracy: 0.9655 - val_loss: 0.2171 - val_accuracy: 0.9420


In [70]:
pred_y = mmodel.predict(valid_x)
y_pred_classes = np.argmax(pred_y,axis=1)

2072/2072 [==============================] - 9s 4ms/step


In [71]:
f1 = f1_score(valid_y, y_pred_classes, average= 'weighted')
f1

0.937412243261704

In [72]:
print(f'Classes: {encoder.classes_}\r\n')

get_classification_report(valid_y, y_pred_classes)

Classes: ['GEOPOLIT' 'LOC' 'MEDIA' 'ORG' 'OUT' 'PER']

              precision    recall  f1-score   support

           0       0.89      0.90      0.89      1119
           1       0.86      0.79      0.83      1145
           2       0.94      0.76      0.84       630
           3       0.91      0.55      0.68      3411
           4       0.94      0.99      0.97     54628
           5       0.97      0.72      0.83      5367

    accuracy                           0.94     66300
   macro avg       0.92      0.79      0.84     66300
weighted avg       0.94      0.94      0.94     66300

CONFUSION MATRIX

col_0     0    1    2     3      4     5
row_0                                   
0      1005   20    1    28     64     1
1        15  910    2    24    192     2
2         1    2  481    13    131     2
3       105   32   19  1867   1372    16
4         5   95   10   119  54320    79
5         0    1    1     3   1491  3871


Обучим нейронную сеть на биграммах и триграммах.

In [73]:
def custom_standardization(input_data):
    # Здесь может быть предобработка текста.
    return input_data

vocab_size = 30000
seq_len = 10

vectorize_layer = TextVectorization(
                            standardize=custom_standardization,
                            max_tokens=vocab_size,
                            output_mode='int',
                            ngrams=(1, 3),
                            output_sequence_length=seq_len)

# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
text_data = train_data.map(lambda x, y: x)
vectorize_layer.adapt(text_data)

In [74]:
mmodel = modelNER()

In [75]:
mmodel.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

In [76]:
mmodel.fit( train_data,
            validation_data=valid_data,
            epochs=3)

Epoch 1/3
12432/12432 [==============================] - 501s 40ms/step - loss: 0.3010 - accuracy: 0.9122 - val_loss: 0.2082 - val_accuracy: 0.9372
Epoch 2/3
12432/12432 [==============================] - 499s 40ms/step - loss: 0.1279 - accuracy: 0.9615 - val_loss: 0.2759 - val_accuracy: 0.8947
Epoch 3/3
12432/12432 [==============================] - 466s 37ms/step - loss: 0.1104 - accuracy: 0.9654 - val_loss: 0.2148 - val_accuracy: 0.9408


In [80]:
pred_y = mmodel.predict(valid_x)
y_pred_classes = np.argmax(pred_y,axis=1)

2072/2072 [==============================] - 9s 4ms/step


In [78]:
f1 = f1_score(valid_y, y_pred_classes, average= 'weighted')
f1

0.9359607765383628

In [79]:
print(f'Classes: {encoder.classes_}\r\n')

get_classification_report(valid_y, y_pred_classes)

Classes: ['GEOPOLIT' 'LOC' 'MEDIA' 'ORG' 'OUT' 'PER']

              precision    recall  f1-score   support

           0       0.89      0.90      0.89      1119
           1       0.85      0.79      0.82      1145
           2       0.93      0.76      0.84       630
           3       0.90      0.53      0.67      3411
           4       0.94      0.99      0.97     54628
           5       0.97      0.72      0.83      5367

    accuracy                           0.94     66300
   macro avg       0.91      0.78      0.84     66300
weighted avg       0.94      0.94      0.94     66300

CONFUSION MATRIX

col_0     0    1    2     3      4     5
row_0                                   
0      1003   21    3    27     64     1
1        17  910    0    21    195     2
2         1    2  481    13    132     1
3       100   37   19  1814   1424    17
4         5   99   12   139  54294    79
5         0    2    1     3   1491  3870


Результьат выше у сети, которая обучалась на N-граммах.